# MiniMax H3 成熟工作流：Google Colab + Drive

本 Notebook 不公开私有 Director 节点。模型、环境快照、队列和成片都保存在 `MyDrive/minimaxH3_colab`。

首次准备请使用免费的 CPU 运行时：它会安装基础环境、下载约 43GB 模型并生成 Drive 环境快照。真正开始生成视频时再切换 GPU；G4/Blackwell 使用 PyTorch 2.11 + CUDA 13.0，并默认使用原生 PyTorch attention。首次 GPU 会话执行 CUDA 验证，以后会话直接恢复快照。

In [ ]:
import platform, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
print('阶段：CPU 下载与环境准备；生成视频前再切换 GPU。')

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/minimaxH3_colab')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if token:
    os.environ['H3_SOURCE_GITHUB_TOKEN'] = token
    print('检测到 GITHUB_TOKEN：仅在首次构建或启用重建开关时刷新私有源包。')
else:
    print('使用 Drive 中预置的私有源包。')
print('Drive root:', DRIVE_ROOT)

In [ ]:
import shutil, subprocess
from pathlib import Path

LAUNCHER = Path('/content/minimaxH3_colab_launcher')
if LAUNCHER.exists():
    shutil.rmtree(LAUNCHER)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/SydneyMCDonaldbigking/minimaxH3_colab.git',
    str(LAUNCHER),
], check=True)

## CPU：安装或恢复环境并下载模型

第一次运行会构建 `/content/h3env + ComfyUI + 私有工作流`，把模型直接保存到 Drive，并生成版本化环境快照。CPU 阶段不安装 SageAttention，也不占用 GPU。

默认只下载实际生产使用的 `ref2va` 模型。以后切到 GPU 再运行本单元时，会恢复快照并完成 CUDA 验证。Blackwell 默认跳过不兼容的通用 SageAttention 包。只有升级工作流、PyTorch 或 ComfyUI 时，才把 `REBUILD_RUNTIME` 改成 `True`。

In [ ]:
import subprocess, sys

REBUILD_RUNTIME = False
command = [
    sys.executable,
    '/content/minimaxH3_colab_launcher/colab/bootstrap_colab.py',
    '--drive-root', '/content/drive/MyDrive/minimaxH3_colab',
    '--profile', 'ref2va',
    '--prepare-on-cpu',
]
if REBUILD_RUNTIME:
    command.append('--rebuild-runtime')
subprocess.run(command, check=True)

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable,
    '/content/minimaxH3_colab_launcher/colab/launch_colab.py',
    '--drive-root', '/content/drive/MyDrive/minimaxH3_colab',
], check=True)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8188, height=900)

## 路径

- 通过机器 QA 的视频：`MyDrive/minimaxH3_colab/review_ready`
- 最终交付：`MyDrive/minimaxH3_colab/deliveries`
- 日志：`MyDrive/minimaxH3_colab/logs`
- 任务队列：`MyDrive/minimaxH3_colab/jobs`